# Inspect a spin-glass run
Run the cells in order. The notebook reads saved results; it does not run the solver or modify your data.

Start Jupyter from the project directory. By default the newest `.npz` file in `Output` is selected. Set `filename` below to inspect a particular run instead.


In [ ]:
from pathlib import Path
import sys

# Run this notebook from the project root.
python_dir = Path("python").resolve()
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))
import json
import numpy as np
import matplotlib.pyplot as plt

# Set a path here to select a specific run, for example:
# filename = Path("Output/trajectory_2026-10-18_232117_772771.npz")
filename = None
output_dir = Path("Output")

files = sorted(output_dir.glob("*.npz"), key=lambda p: (p.stat().st_mtime_ns, p.name))
print("Available files:")
for path in files:
    print(" ", path)
if filename is None:
    if not files:
        raise FileNotFoundError("No .npz files in Output. Run either program first or set filename explicitly.")
    filename = files[-1]
filename = Path(filename)
print("Selected:", filename)


## Load arrays and inspect parameters

In [ ]:
with np.load(filename, allow_pickle=False) as data:
    print("Contents:", data.files)
    for name in data.files:
        print(f"{name}: shape={data[name].shape}, dtype={data[name].dtype}")
    is_simulation = "run_type" in data.files and data["run_type"].item() == "spin_simulation"
    columns = data["trajectory_columns"].tolist() if "trajectory_columns" in data.files else ["time", "m", "q", "mass", "residual"]
    simulation = json.loads(data["simulation"].item()) if is_simulation else None
    outside_counts = data["outside_counts"].copy() if is_simulation else None
    parameters = json.loads(data["parameters"].item())
    trajectory = data["trajectory"].copy()
    trajectories = data["trajectories"].copy() if "trajectories" in data.files else None
    ensemble = json.loads(data["ensemble"].item()) if "ensemble" in data.files else None
    run_seeds = data["run_seeds"].copy() if "run_seeds" in data.files else None
    run_outside_counts = data["run_outside_counts"].copy() if "run_outside_counts" in data.files else None
    rho = data["rho"].copy()
    h = data["h"].copy()
    energy = data["energy"].copy() if "energy" in data.files else None
    solver = json.loads(data["solver"].item()) if "solver" in data.files else None
    saddle_residuals = data["saddle_residuals"].copy() if "saddle_residuals" in data.files else None
    saddle_names = data["saddle_residual_names"].copy() if "saddle_residual_names" in data.files else None
    saddle_times = data["saddle_times"].copy() if "saddle_times" in data.files else None

print("\nRun parameters:")
print(json.dumps(parameters, indent=2))
# Zero-duration runs save an empty trajectory with shape (0,).
if trajectory.size == 0:
    trajectory = np.empty((0, 5))
if trajectory.ndim != 2 or trajectory.shape[1] != 5:
    raise ValueError(f"Expected five trajectory columns, got {trajectory.shape}")

if is_simulation and trajectories is not None:
    if trajectories.ndim != 3 or trajectories.shape[1:] != trajectory.shape:
        raise ValueError("Expected trajectories with shape (runs, rows, 5)")
    print("Ensemble:", json.dumps(ensemble, indent=2))
    print("All trajectory seeds:", run_seeds)
    print("Legacy arrays below describe run 0; plots overlay every run.")
    print("Final outside counts for every run:", run_outside_counts[:, -1])

if is_simulation:
    print("Spin simulation:", json.dumps(simulation, indent=2))
    print("No replica q is measured. Time is attempted updates per spin.")
    print("Final histogram underflow/overflow counts (rows +1, -1):", outside_counts[-1])
elif solver is None:
    print("Legacy file: trajectory residual is the density RMS error.")
else:
    print("\nSolver:", json.dumps(solver, indent=2))
    if saddle_residuals is not None:
        print("Saddle equation residual columns:", saddle_names)
        print("Saddle solve times (first 10):", saddle_times[:10])
        print("Equation residuals (first 10):")
        print(saddle_residuals[:10])

# Appendix D: energy per spin uses the full field h, without subtracting j0*m.
if is_simulation:
    final_energy = energy[-1]  # Exact spin/field observable, independent of histogram range.
else:
    dh = h[1] - h[0]
    final_energy = -0.5 * dh * np.dot(h, rho[0] - rho[1])
print("Final energy per spin:", final_energy)
if energy is not None and energy.shape != (len(trajectory),):
    raise ValueError("Energy must have one value per trajectory row")
if energy is None:
    print("Legacy file: only final energy is available; rerun the solver for an energy trajectory.")

if parameters.get("run_status") == "failed":
    print("PARTIAL TRAJECTORY: solver failed at t=", parameters["completed_time"])
    print(parameters["failure_message"])


## Inspect numerical data
Finite-difference columns are time, magnetization, saddle `q`, mass and residual. Its `q` and residual refer to the saddle before the density update; magnetization and energy refer to the updated density.

Spin simulation columns are time, magnetization, energy, `dm/dt` and `dE/dt`. Derivatives use consecutive saved rows; the initial derivatives are undefined (`NaN`). No replica `q` is measured. `rho` is a joint spin/field histogram; underflow and overflow counts report any mass outside the chosen range.

For ensemble archives, `trajectories[run_index]` contains each complete trajectory. The legacy arrays (`trajectory`, `rho`, etc.) refer to run index 0, not an average. Run indices and seeds have matching order.


In [ ]:
print("Trajectory columns:", columns)
print("Rows:", len(trajectory))
print(trajectory[:10])  # Change to print(trajectory) to inspect more rows.
print("\nField grid (first 10 cells):")
print(h[:10])
print("\nFinal density (first 10 cells; rows: plus, minus):")
print(rho[:, :10])

if trajectories is not None:
    print("All trajectories shape (runs, rows, columns):", trajectories.shape)
    # Example: trajectories[1, :10] inspects the first ten rows of run index 1.
    print("Final [time, m, energy] for every run:")
    print(trajectories[:, -1, :3])


## Plot the trajectory
Both run types show magnetization and energy per spin. Finite-difference results also show recorded saddle `q`. Older finite-difference files without an energy history show only final energy.

The figure is saved as a 200 dpi PNG in the project-level `figures` folder alongside `Output`. Replotting the same archive replaces its figure.

For simulation ensembles, every m and E trajectory is overlaid in the same colour with line width `max(0.5, 2.0 / n_runs)` points, without markers. Darker concentrations indicate overlapping paths. These show variability for **fixed couplings**, with independently seeded initial spins and updates; they are not confidence intervals. A minimum width of 0.5 points keeps individual paths visible even for large ensembles.

For k=0, black dotted lines mark RS equilibrium m and E (thesis 6.19–6.20), using only the positive magnetisation branch, independently of m0. These are infinite-system RS references, not a guarantee of convergence. k>0 is not covered.


In [ ]:
import importlib
import combined_runs
importlib.reload(combined_runs)

if len(trajectory) == 0:
    print("No trajectory steps to plot (for example, t_end = 0). Final density is available above.")
else:
    fig, axes = plt.subplots(2 if is_simulation else 3, 1, figsize=(9, 6 if is_simulation else 9), sharex=True, constrained_layout=True)
    if is_simulation and trajectories is not None:
        n_runs = len(trajectories)
        line_width = combined_runs.simulation_linewidth(n_runs)
        print(f"Simulation plot: {n_runs} runs; line width={line_width:g} pt; {filename}")
        for index, tr in enumerate(trajectories):
            axes[0].plot(tr[:, 0], tr[:, 1], color="tab:blue", linewidth=line_width,
                         label=f"Magnetization ({n_runs} runs)" if index == 0 else None)
            axes[-1].plot(tr[:, 0], tr[:, 2], color="tab:green", linewidth=line_width,
                          label=f"Energy per spin ({n_runs} runs)" if index == 0 else None)
    else:
        axes[0].plot(trajectory[:, 0], trajectory[:, 1], ".-", markersize=1, label="Magnetization")
    axes[0].set_ylabel("Magnetization m")
    if not is_simulation:
        axes[1].plot(trajectory[:, 0], trajectory[:, 2], ".-", color="tab:orange", markersize=1, label="Recorded saddle q")
        axes[1].set_ylabel("Saddle q")
    if is_simulation and trajectories is not None:
        pass  # All energy trajectories are plotted above.
    elif energy is not None:
        axes[-1].plot(trajectory[:, 0], energy, ".-", color="tab:green", markersize=1, label="Energy per spin")
    else:
        axes[-1].plot(trajectory[-1, 0], final_energy, "o", color="tab:green", markersize=1, label="Final energy only (legacy file)")
    axes[-1].set_ylabel("Energy per spin E")
    axes[-1].set_xlabel("Simulation time")
    import rs_equilibrium
    importlib.reload(rs_equilibrium)
    rs_equilibrium.add_equilibrium_references(axes[0], axes[-1], [parameters])
    for ax in axes:
        ax.grid(True, alpha=0.3)
        ax.legend()
    fig.suptitle(filename.name)
    if parameters.get("run_status") == "failed":
        fig.suptitle(f"Partial trajectory — solver failed at t={parameters['completed_time']:.6g}")
    figures_dir = Path("figures").resolve()
    figures_dir.mkdir(parents=True, exist_ok=True)
    figure_path = figures_dir / f"{filename.stem}_trajectory.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    print("Saved figure:", figure_path)
    plt.show()
